In [1]:
from IPython.core.interactiveshell import InteractiveShell
InteractiveShell.ast_node_interactivity = "all"

In [2]:
# pip install python-dotenv

In [3]:
from pathlib import Path
import os

import duckdb
import pandas as pd
import requests
from dotenv import load_dotenv

In [4]:
PROJECT_ROOT = Path.cwd()
DUCKDB_PATH = PROJECT_ROOT / "data" / "warehouse" / "project.duckdb"

duckdb_con = duckdb.connect(str(DUCKDB_PATH))
print("Connected to:", DUCKDB_PATH)

Connected to: /Users/lingzitong/Desktop/MSIN0166 Individual Assignment/data/warehouse/project.duckdb


In [5]:
high_entities = duckdb_con.execute("""
    SELECT entity_id, entity_name
    FROM entity_risk_signals_v2
    WHERE review_priority_band = 'High'
""").fetchdf()

high_entities

,entity_id,entity_name
0,16392049,AD3 PROPERTIES LTD
1,16484242,ALOEHAWK LIMITED
2,16338097,ALTITUDE UK PROPERTIES LTD
3,SC859869,GILLESPIE PROPERTY SCOTLAND LIMITED
4,SC865801,GOSHEN DOM PROPERTIES LTD
...,...,...
193,16536219,WHY CAMBER 1 LTD
194,16430217,WILSON-BYRNE PROPERTY LTD
195,16537084,WOOD GREEN DEVELOPERS LTD
196,16517513,WP 29 PROPERTY LTD


In [6]:
load_dotenv()

API_KEY = os.getenv("CompaniesHouse_API_KEY")
BASE_URL = "https://api.company-information.service.gov.uk"
if not API_KEY:
    raise ValueError("CompaniesHouse_API_KEY is not set.")

True

In [7]:
def get_company_profile(company_number: str) -> dict:
    url = f"{BASE_URL}/company/{company_number}"
    response = requests.get(url, auth=(API_KEY, ""))
    response.raise_for_status()
    return response.json()

profiles = []

for _, row in high_entities.iterrows():
    company_number = str(row["entity_id"]).strip()
    try:
        profile = get_company_profile(company_number)
        profiles.append({
            "entity_id": company_number,
            "entity_name": row["entity_name"],
            "company_status_api": profile.get("company_status"),
            "date_of_creation_api": profile.get("date_of_creation"),
            "has_insolvency_history": profile.get("has_insolvency_history"),
            "type_api": profile.get("type"),
            "jurisdiction": profile.get("jurisdiction"),
        })
    except Exception as error:
        profiles.append({
            "entity_id": company_number,
            "entity_name": row["entity_name"],
            "api_error": str(error),
        })

profiles_df = pd.DataFrame(profiles)
profiles_df

,entity_id,entity_name,company_status_api,date_of_creation_api,has_insolvency_history,type_api,jurisdiction
0,16392049,AD3 PROPERTIES LTD,active,2025-04-16,False,ltd,england-wales
1,16484242,ALOEHAWK LIMITED,active,2025-05-30,False,ltd,england-wales
2,16338097,ALTITUDE UK PROPERTIES LTD,active,2025-03-24,False,ltd,england-wales
3,SC859869,GILLESPIE PROPERTY SCOTLAND LIMITED,active,2025-08-21,False,ltd,scotland
4,SC865801,GOSHEN DOM PROPERTIES LTD,active,2025-10-09,False,ltd,scotland
...,...,...,...,...,...,...,...
193,16536219,WHY CAMBER 1 LTD,active,2025-06-23,False,ltd,england-wales
194,16430217,WILSON-BYRNE PROPERTY LTD,active,2025-05-06,False,ltd,england-wales
195,16537084,WOOD GREEN DEVELOPERS LTD,active,2025-06-23,False,ltd,england-wales
196,16517513,WP 29 PROPERTY LTD,active,2025-06-13,False,ltd,england-wales


In [8]:
profiles_df.to_csv("data/processed/entity_external_enrichment.csv", index=False)

In [9]:
duckdb_con.register("profiles_api_df", profiles_df)

duckdb_con.execute("""
CREATE OR REPLACE TABLE entity_external_enrichment AS
SELECT *
FROM profiles_api_df
""")

In [10]:
duckdb_con.execute("""
    SELECT *
    FROM entity_external_enrichment
    LIMIT 5
""").fetchdf()

,entity_id,entity_name,company_status_api,date_of_creation_api,has_insolvency_history,type_api,jurisdiction
0,16392049,AD3 PROPERTIES LTD,active,2025-04-16,False,ltd,england-wales
1,16484242,ALOEHAWK LIMITED,active,2025-05-30,False,ltd,england-wales
2,16338097,ALTITUDE UK PROPERTIES LTD,active,2025-03-24,False,ltd,england-wales
3,SC859869,GILLESPIE PROPERTY SCOTLAND LIMITED,active,2025-08-21,False,ltd,scotland
4,SC865801,GOSHEN DOM PROPERTIES LTD,active,2025-10-09,False,ltd,scotland


In [11]:
duckdb_con.close()
print("DuckDB connection closed.")

DuckDB connection closed.
